# Tentativo 19 · 5. ModernBERT addestrato

ModernBERT si addestra tutto, un fold alla volta, sul compito scelto, con lo split per voto, come nei tentativi 16 e 18:
5 epoche, lr 3·10⁻⁵, batch effettivo 32, nessuna validazione (si tiene l'ultima epoca).
- **Classificazione**: 5 uscite, entropia incrociata con le classi bilanciate e label smoothing 0,1.
- **Regressione**: un'uscita, errore quadratico sul voto/100 (l'uscita si rimoltiplica per 100).

Ogni articolo entra intero (`MAX_TOKEN` = 4.096, sopra l'articolo più lungo: notebook 3). Si salvano i logit o il voto
previsto per ogni articolo di ogni fold, la curva della loss e il modello di ogni fold (`Modelli/19_DatasetAnonimo/`).

In [ ]:
COMPITO = "classificazione"
MODELLO = "answerdotai/ModernBERT-base"
NOME = "ModernBERT"
MAX_TOKEN = 4096
EPOCHE = 5
LR = 3e-5
BATCH = 4
GB_GPU = 16

In [ ]:
import time
import warnings

import pandas as pd

import modelli as Mo

warnings.filterwarnings("ignore")
OUT = Mo.Path("Results"); OUT.mkdir(exist_ok=True)
def registra(msg, file=OUT / "avanzamento.log"):
    riga = f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {msg}"
    print(riga, flush=True)
    with open(file, "a") as f:
        f.write(riga + "\n")
art, t = Mo.leggi()
print(f"{COMPITO}: {len(art):,} articoli, {len(t)} testate"); print(Mo.ambiente())

In [ ]:
for k in range(1, 6):
    Mo.bert_fold(COMPITO, art, k, MODELLO, NOME, registra, max_token=MAX_TOKEN, epoche=EPOCHE, lr=LR, batch=BATCH, gb_gpu=GB_GPU)

In [ ]:
curve = pd.concat([pd.read_csv(f).assign(fold=f.stem[-1]) for f in sorted((Mo.PREVISIONI / COMPITO / NOME).glob("curva_fold*.csv"))])
curve.pivot(index="epoca", columns="fold", values="loss").round(4)